0. 준비: 라이브러리 설치

In [ ]:
pip install cryptography

1. RSA 공개키/개인키 생성 코드

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

def generate_keys():
    # 2048비트 RSA 키 생성
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )

    public_key = private_key.public_key()

    # 개인키를 PEM 형식으로 저장 (비밀번호 없이 예시)
    with open("private_key.pem", "wb") as f:
        f.write(
            private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.NoEncryption(),
            )
        )

    # 공개키를 PEM 형식으로 저장
    with open("public_key.pem", "wb") as f:
        f.write(
            public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo,
            )
        )

    print("키 생성 완료: private_key.pem, public_key.pem")

키 생성 완료: private_key.pem, public_key.pem


2.1. 파일 전송

In [6]:
# 서버가 암호화해서 전송할 파일 경로
path = "API_KEY.env"

In [ ]:
import socket
import struct
import os
from cryptography.hazmat.primitives.asymmetric import padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# 서버 실행
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind(("0.0.0.0", 5000))
server.listen(1)

print("서버 대기중...")

conn, addr = server.accept()
print(f"클라이언트 접속: {addr}")

# ================================
# 1) 클라이언트 공개키 수신
# ================================
key_len_bytes = conn.recv(4)
key_len = struct.unpack(">I", key_len_bytes)[0]
client_public_key_bytes = conn.recv(key_len)

client_public_key = serialization.load_pem_public_key(client_public_key_bytes)

print("클라이언트 공개키 수신 완료")

# ================================
# 2) 파일 읽기
# ================================
file_name = os.path.basename(path)
file_name_bytes = file_name.encode("utf-8")

with open(path, "rb") as f:
    file_data = f.read()

# ================================
# 3) AES 키 생성 + 파일 암호화
# ================================
aes_key = AESGCM.generate_key(bit_length=256)
aesgcm = AESGCM(aes_key)
nonce = os.urandom(12)

encrypted_file = aesgcm.encrypt(nonce, file_data, None)

# ================================
# 4) AES 키를 RSA 공개키로 암호화
# ================================
encrypted_aes_key = client_public_key.encrypt(
    aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

# ================================
# 5) 파일명 + AES키 + nonce + 암호문 전송
# ================================
# 파일명 정보
conn.sendall(struct.pack(">I", len(file_name_bytes)))
conn.sendall(file_name_bytes)

# RSA로 암호화된 AES 키
conn.sendall(struct.pack(">I", len(encrypted_aes_key)))
conn.sendall(encrypted_aes_key)

# nonce
conn.sendall(nonce)

# 대칭키로 암호화된 파일
conn.sendall(struct.pack(">I", len(encrypted_file)))
conn.sendall(encrypted_file)

print("파일 전송 완료")
conn.close()
server.close()


서버 대기중...
클라이언트 접속: ('192.168.75.62', 50594)
클라이언트 공개키 수신 완료


ValueError: Encryption failed

2.2. 파일 수신

In [ ]:
SERVER_IP = "192.168.75.182"

In [ ]:
import socket
import struct
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.ciphers.aead import AESGCM

# ==========================
# 접속할 서버 IP 입력
# ==========================
PORT = 5000

# ==========================
# 1. 클라이언트 RSA 키 생성
# ==========================
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

# 공개키 PEM 직렬화
public_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
)

public_key_len = len(public_pem)

# ==========================
# 2. 서버 접속
# ==========================
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((SERVER_IP, PORT))
print("서버 접속 완료")

# ==========================
# 3. 클라이언트 공개키 전송
# ==========================
client.sendall(struct.pack(">I", public_key_len))
client.sendall(public_pem)
print("클라이언트 공개키 전송 완료")

# ==========================
# 4. 서버로부터 파일명 수신
# ==========================
name_len = struct.unpack(">I", client.recv(4))[0]
file_name = client.recv(name_len).decode("utf-8")
print(f"받은 파일명: {file_name}")

# ==========================
# 5. RSA로 암호화된 AES 키 수신
# ==========================
aes_key_len = struct.unpack(">I", client.recv(4))[0]
encrypted_aes_key = client.recv(aes_key_len)

# AES 키 복호화 (RSA private key)
aes_key = private_key.decrypt(
    encrypted_aes_key,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
print("AES 키 복호화 완료")

# ==========================
# 6. nonce 수신
# ==========================
nonce = client.recv(12)  # AES-GCM 표준 nonce 길이
print("nonce 수신 완료")

# ==========================
# 7. 암호문 파일 내용 수신
# ==========================
encrypted_file_len = struct.unpack(">I", client.recv(4))[0]
encrypted_file = client.recv(encrypted_file_len)
print(f"암호문 파일 수신 완료 ({encrypted_file_len} bytes)")

client.close()

# ==========================
# 8. AES-GCM으로 파일 복호화
# ==========================
aesgcm = AESGCM(aes_key)
plaintext = aesgcm.decrypt(nonce, encrypted_file, None)
print("파일 복호화 완료!")

# ==========================
# 9. 파일 저장
# ==========================
with open(file_name, "wb") as f:
    f.write(plaintext)

print(f"복호화된 파일 저장 완료 → {file_name}")
